# LD-4 — fusion DETR + B3+λ, inférence seule (GPU)

In [ ]:

import time
T_START = time.monotonic()          # pris au tout début de la cellule
import datetime, faulthandler, hashlib, json, os, shlex, shutil, subprocess, sys, zipfile
from pathlib import Path
faulthandler.enable()

def _utc():
    return datetime.datetime.now(datetime.timezone.utc).strftime('%H:%M:%S')

print('ld4 start', _utc(), flush=True)
OUT = Path('/kaggle/working')
KIT_CANDIDATES = ['/kaggle/input/babaorum-ld4-kit', '/kaggle/input/datasets/mabaku/babaorum-ld4-kit']
PAGES_CANDIDATES = ['/kaggle/input/babaorum-e2e3-run3-fp-classes/e2e3/mf', '/kaggle/input/notebooks/mabaku/babaorum-e2e3-run3-fp-classes/e2e3/mf']
CKPT_CANDIDATES = ['/kaggle/input/babaorum-e2e5-lambda/logs/checkpoint_last.pt', '/kaggle/input/notebooks/mabaku/babaorum-e2e5-lambda/logs/checkpoint_last.pt']
LD2_CANDIDATES = ['/kaggle/input/babaorum-ld2-run/ld2', '/kaggle/input/notebooks/mabaku/babaorum-ld2-run/ld2']
E2E4_CANDIDATES = ['/kaggle/input/babaorum-e2e4-extraction/e2e4_extraction_report.json', '/kaggle/input/notebooks/mabaku/babaorum-e2e4-extraction/e2e4_extraction_report.json']
BUNDLE_CANDIDATES = ['/kaggle/input/babaorum-stage-s-bundle-v5/stage_s_bundle_v5.zip.bin', '/kaggle/input/datasets/mabaku/babaorum-stage-s-bundle-v5/stage_s_bundle_v5.zip.bin']
E2E5_SHA = 'ef5e826803c347f73e49c83c0f21f61399f9ea06ee985905bd17644dff3844fa'
LD2_SHA = 'e5afd1432e10aa8e06191678844ae350348f7586f505d1a766aba3a5a44b42c3'
LD2_FILE = 'ld2_main_step003271.pt'
E2E4_SHA = '714bc571db6a4e394027ec4e98cd4c997bde5ff54c735fbc8f32d9cdc4218654'
BUNDLE_SHA = 'ca41c09272248c5096f2547fe738eee622db5af50414182537ac8e94e43367f5'
COMMIT = '23875ab05425ce2aa07742bf314d02436c2f5642'
REQUIRED_PIP = ['Pillow>=12.3,<13', 'fonttools>=4.60,<5', 'jsonschema>=4.25,<5', 'lxml>=6,<7', 'numpy>=2.3,<3', 'shapely>=2.1,<3']
TIMEOUTS = {'pip': 1500, 'unzip': 900, 'clone': 600, 'controls': 600, 'D': 900, 'CAL': 1800, 'generate': 1500}
LIMITS = {'deadline_s': 5400, 'stop_margin_s': 120, 'eval_margin_s': 300, 'estimate_factor': 1.5, 'generation_estimate_s': 600, 'heartbeat_s': 60}
RUN_FILES = ['scripts/eval_ld4.py', 'config/ld4.json', 'tests/test_ld4.py', 'scripts/data3_generate.py']
def resolve_one(candidates, marker):
    """Exactement un candidat dont <candidat>/<marqueur> existe. Aucun listage."""
    hits = [Path(c) for c in candidates if (Path(c) / marker).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"montage : {len(hits)} candidat(s) pour {marker}")
    return hits[0]

def resolve_file(candidates):
    """Exactement un candidat qui est un fichier (§12 (15))."""
    hits = [Path(c) for c in candidates if Path(c).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"fichier : {len(hits)} candidat(s)")
    return hits[0]


def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

KIT = resolve_one(KIT_CANDIDATES, "config/ld4.json")
PAGES = resolve_one(PAGES_CANDIDATES, "pages/mf_0000.json")
CKPT = resolve_file(CKPT_CANDIDATES)
LD2DIR = resolve_one(LD2_CANDIDATES, "ld2_manifest.json")
E2E4 = resolve_file(E2E4_CANDIDATES)
BUNDLE_ZIP = resolve_file(BUNDLE_CANDIDATES)
assert sha256(CKPT) == E2E5_SHA, "checkpoint E2E-5 : sha différent"
assert sha256(LD2DIR / LD2_FILE) == LD2_SHA, "checkpoint LD-2 : sha différent"
assert sha256(E2E4) == E2E4_SHA, "rapport e2e4 : sha différent"
assert sha256(BUNDLE_ZIP) == BUNDLE_SHA, "bundle v5 : sha différent"
sys.path.insert(0, str(KIT / "src"))
sys.path.insert(0, str(KIT / "scripts"))
import importlib.util as _u
_spec = _u.spec_from_file_location("data2_common", KIT / "scripts" / "data2_common.py")
C = _u.module_from_spec(_spec); _spec.loader.exec_module(C)
import ld2_core as C2

BUNDLE_DIR = Path("/tmp/ld4_bundle")
SRC = Path("/tmp/ld4_heritage")
LIBS = Path("/tmp/ld4_genlib")
WORK = Path("/tmp/ld4_work")

def _argv(phase, extra=()):
    return [sys.executable, str(KIT / "scripts" / "eval_ld4.py"), "--phase", phase, "--root", str(PAGES),
            "--out", str(RUN), "--ld2-dir", str(LD2DIR), "--e2e5-checkpoint", str(CKPT), "--e2e4-report", str(E2E4),
            "--gen-src", str(SRC), "--work", str(WORK), "--extra-libs", str(LIBS), "--cal-out", str(CAL_OUT),
            "--device", "cuda", "--cache-dir", "/tmp/ld4_hf", *extra]

def d_mille_check():
    """Les 51 fichiers suivis du générateur cloné égalent bundle_manifest.json (comme DATA-2)."""
    manifest = json.loads((BUNDLE_DIR / "bundle_manifest.json").read_text())
    for rel, want in manifest["mille_feuilles_sha256"].items():
        assert sha256(SRC / rel[len("heritage-synth/"):]) == want, ("fichier du générateur différent", rel)
    print("générateur : fichiers contrôlés", len(manifest["mille_feuilles_sha256"]), flush=True)

GEN_CMDS = {
    "pip": [sys.executable, "-m", "pip", "install", "--quiet", "--timeout", "30", "--retries", "1", "--target", str(LIBS),
            *REQUIRED_PIP],
    "unzip": [sys.executable, "-c", "import sys, zipfile; zipfile.ZipFile(sys.argv[1]).extractall(sys.argv[2])",
              str(BUNDLE_ZIP), str(BUNDLE_DIR)],
    "clone": ["git", "clone", "-q", str(BUNDLE_DIR / "heritage-synth.gitbundle"), str(SRC)],
    "checkout": ["git", "-C", str(SRC), "checkout", "-q", COMMIT],
}
GEN_CAPS = {"pip": TIMEOUTS["pip"], "unzip": TIMEOUTS["unzip"], "clone": TIMEOUTS["clone"], "checkout": TIMEOUTS["clone"]}
# Préparation à plafonds FIXES : utilisée telle quelle par la FUMÉE (CPU, aucune échéance globale de cellule à tenir). Le run
# complet reconstruit ces quatre étapes avec step_argv (plafond = restant - marge d'arrêt).
GEN_PREP = [*({"name": n, "argv": GEN_CMDS[n], "timeout_s": GEN_CAPS[n]} for n in ("pip", "unzip", "clone", "checkout")),
            {"name": "mille_check", "fn": d_mille_check}]

RUN = OUT / "ld4"
CAL_OUT = OUT / "ld4_cal"
EXITS = OUT / "exits"
class Incomplete(Exception):
    pass

def remaining():
    return LIMITS["deadline_s"] - (time.monotonic() - T_START)

def allowed(cap, reserve=0.0):
    """max(0, min(plafond, restant - réserve))."""
    return C2.allowed(cap, remaining(), reserve)

def step_argv(ph, build_argv, cap, reserve=0.0, **kw):
    def argv():
        t = allowed(cap, reserve)
        if t <= 0:
            raise Incomplete(f"échéance globale atteinte avant l'étape {ph}")
        st["timeout_s"] = t
        return build_argv()
    st = {"name": ph, "argv": argv, "timeout_s": cap, **kw}
    return st

def guard_fn(name, fn):
    """Étape fn : refusée (INCOMPLETE) si l'échéance globale moins la marge d'arrêt est atteinte, sans exécuter fn."""
    def run():
        if remaining() - LIMITS["stop_margin_s"] <= 0:
            raise Incomplete(f"échéance globale atteinte avant l'étape {name}")
        return fn()
    return {"name": name, "fn": run}

def s_estimate():
    """Porte avant EVAL : (chargements + n pages x durée mesurée par CAL sur une page d'ENTRAÎNEMENT) x facteur + génération.
    Au-delà du temps restant moins la marge : INCOMPLETE, avant que les pages soient générées ou lues."""
    cal = json.loads((CAL_OUT / "ld4_cal.json").read_text())
    avail = remaining() - LIMITS["eval_margin_s"]
    est = cal["estimate_s"] * LIMITS["estimate_factor"] + LIMITS["generation_estimate_s"]
    (OUT / "ld4_eval_estimate.json").write_text(json.dumps(
        {"estimate_s": est, "factor": LIMITS["estimate_factor"], "generation_estimate_s": LIMITS["generation_estimate_s"],
         "remaining_s": remaining(), "margin_s": LIMITS["eval_margin_s"], "available_s": avail, "cal": cal}, indent=1))
    print(f"ESTIMATION_EVAL_S {est:.0f} ; disponible {avail:.0f}", flush=True)
    if est > avail:
        raise Incomplete(f"budget insuffisant : estimation {est:.0f} s > {avail:.0f} s restants")

def eval_argv():
    t = allowed(LIMITS["deadline_s"], LIMITS["eval_margin_s"])
    if t <= 0:
        raise Incomplete("échéance globale atteinte avant EVAL")
    EVAL_STEP["timeout_s"] = t
    return _argv("EVAL")

def ld4_cleanup(rec):
    """TIMEOUT, échéance ou échec d'étape : INCOMPLETE à l'étape, sauf si l'étape a écrit FAILED (refus, garde)."""
    if rec["status"] == "TIMEOUT" or str(rec.get("exception") or "").startswith("Incomplete:"):
        rec["status"] = "INCOMPLETE"
    elif rec["status"] == "FAILED" and rec["exit_code"] != "EXCEPTION":
        f = EXITS / f"ld4_run_exit_{rec['step']}.json"
        if not f.is_file() or json.loads(f.read_text()).get("status") != "FAILED":
            rec["status"] = "INCOMPLETE"

def s_pre():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA indisponible : run complet refusé")
    from importlib.metadata import version as _v
    (OUT / "ld4_env.json").write_text(json.dumps({
        "python": sys.version, "torch": torch.__version__, "cuda": torch.version.cuda, "numpy": _v("numpy"),
        "PIL": _v("pillow"), "transformers": _v("transformers"), "huggingface_hub": _v("huggingface_hub")}, indent=1))
    for d in (BUNDLE_DIR, SRC, LIBS, WORK):
        shutil.rmtree(d, ignore_errors=True)
    print("pre ok", _utc(), flush=True)

OFFLINE = lambda: dict(os.environ, HF_HUB_OFFLINE="1")       # hors ligne sauf pip et D
# TOUTE étape argv est bornée par l'échéance globale : délai = max(0, min(plafond, restant - marge d'arrêt)), calculé au
# lancement ; non positif -> Incomplete SANS lancer le sous-processus. La marge d'arrêt (stop_margin_s, 120 s) couvre le délai de
# grâce du kill de execute_pipeline (SIGTERM puis SIGKILL après 10 s + 5 s d'attente) et le nettoyage final (rmtree, écriture
# de ld4_run_exit.json). EVAL garde eval_margin_s (300 s).
MARGIN = LIMITS["stop_margin_s"]
GEN_FULL = [step_argv(n, (lambda n=n: GEN_CMDS[n]), GEN_CAPS[n], MARGIN) for n in ("pip", "unzip", "clone", "checkout")]
CONTROLS_STEP = step_argv("controls", lambda: _argv("controls"), TIMEOUTS["controls"], MARGIN, env=OFFLINE)
D_STEP = step_argv("D", lambda: _argv("D"), TIMEOUTS["D"], MARGIN)
CAL_STEP = step_argv("CAL", lambda: _argv("CAL"), TIMEOUTS["CAL"], MARGIN, env=OFFLINE)
EVAL_STEP = {"name": "EVAL", "argv": eval_argv, "timeout_s": LIMITS["deadline_s"], "env": OFFLINE}
STEPS = [guard_fn("pre", s_pre), *GEN_FULL, guard_fn("mille_check", lambda: d_mille_check()), D_STEP, CONTROLS_STEP, CAL_STEP,
         guard_fn("estimate", s_estimate), EVAL_STEP]
EXIT = None
try:
    EXIT = C.execute_pipeline(OUT, STEPS, heartbeat_interval_s=LIMITS["heartbeat_s"], cleanup=ld4_cleanup)
finally:
    _rec = dict(EXIT) if EXIT is not None else {"status": "FAILED_BEFORE_EXIT"}
    _rec["kit_sha256"] = {rel: sha256(KIT / rel) for rel in RUN_FILES if (KIT / rel).is_file()}      # fichiers de LD-4 lus
    (OUT / "ld4_run_exit.json").write_text(json.dumps(_rec, indent=1, default=str))
    for d in (BUNDLE_DIR, SRC, LIBS, WORK):                          # aucune image générée ne reste dans la sortie
        shutil.rmtree(d, ignore_errors=True)
print("EXIT", json.dumps({k: EXIT[k] for k in ("status", "step", "exit_code")}) if EXIT else "none", flush=True)
if (RUN / "ld4_report.json").is_file():
    print("REPORT_SHA256", sha256(RUN / "ld4_report.json"), flush=True)
